In [ ]:
''This code runs the Holt-Winters forecasting method and performs some analysis in between.
It is a classic method, and its purpose is to test the forecast of raw data before moving to machine learning.
The cells are commented, but the code should read as a follow-up of my thinking and rethinking during execution.
Because this was exploratory work, I put more effort into generating and analyzing outputs than into cleaning the code for readability.

In [ ]:
# === Cell 0: Libraries ===
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
#from google.colab import files
from statsmodels.tsa.statespace.sarimax import SARIMAX

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from scipy.stats import normaltest # Added for normality test
from statsmodels.tsa.seasonal import STL # Added for STL decomposition
from statsmodels.tsa.stattools import adfuller # Added for stationarity test
%pip install xlsxwriter
import xlsxwriter # Added for writing to Excel
import io # Import io module
import os # Import os module to manage temporary files


In [ ]:
# === Cell 1: EDA ===  

# Clean up potential temporary files from previous runs
temp_plot_names = ['hist_boxplot', 'timeseries_trend', 'rolling_mean_std', 'acf_pacf', 'stl_decomposition']
for plot_name in temp_plot_names:
    temp_filename = f"{plot_name}.png"
    if os.path.exists(temp_filename):
        os.remove(temp_filename)

# === Step 1: Upload file ===
filename = 'b100_structured.csv' # <--- Specify your dataset name here
df = pd.read_csv(filename) #full_comsumption.csv


# === Step 2: Prepare data ===
# Ensure date column is datetime and set as index
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df = df.dropna(subset=['date'])
df.set_index('date', inplace=True)

# --- User Input: Specify the target column name ---
proxy_col = 'b100_structured'  # <--- Specify your target column name here

# Select only the target column for further processing
df = df[[proxy_col]]

# Handle duplicate index values before setting frequency
df = df[~df.index.duplicated(keep='last')]

# Set daily frequency and forward fill all gaps, including weekends
df = df.asfreq('D')  # 'D' = daily frequency
df[proxy_col] = df[proxy_col].fillna(method='ffill') # Forward fill all NaNs created by asfreq or original gaps

# Get first and last date of the index
first_date = df.index.min().strftime('%Y-%m-%d')
last_date = df.index.max().strftime('%Y-%m-%d')


# === Step 2.5: EDA (print + plots) ===
series = df[proxy_col].dropna() # Use dropna for EDA calculations
stats = series.describe(percentiles=[.25,.5,.75])
miss = series.isna().sum()
nz = series # Use the series with potential NaNs for plotting asfreq results correctly
norm_p = normaltest(series.dropna()).pvalue if len(series.dropna()) > 7 else np.nan # Use series for normality test


# STL trend (annual business cycle). If too short, fall back to 65
period = 365 if len(series.dropna()) >= 300 else 65
try: # Added try-except for STL in case of short series
    stl = STL(series.dropna(), period=period).fit() # Use series.dropna() for STL
    trend = stl.trend
    seasonal = stl.seasonal # Get seasonal component
    residual = stl.resid # Get residual component
    slope = (trend.iloc[-1] - trend.iloc[0]) / len(trend)
except:
    trend = None
    seasonal = None # Added for STL decomposition
    residual = None # Added for STL decomposition
    slope = np.nan
    print("STL decomposition failed, potentially due to short series.")


print("\n=== EDA SUMMARY ===")
print(f"Data Range: {first_date} to {last_date}") # Print data range
print(stats)
print(f"Missing: {miss}")
print(f"Normality p-value: {norm_p:.4f}" if not np.isnan(norm_p) else "Normality p-value: n/a")
print(f"STL period used: {period}")
print(f"Trend slope per step: {slope:.6f}" if not np.isnan(slope) else "Trend slope per step: n/a")

# Check for stationarity using ADF test
adf_test = adfuller(series.dropna()) # Use series.dropna() for ADF test
print(f"ADF Statistic: {adf_test[0]:.4f}")
print(f"p-value: {adf_test[1]:.4f}")
print("Is stationary: %s" % ('Yes' if adf_test[1] <= 0.05 else 'No'))


# Prepare Excel file
output_filename_excel = filename.replace('.csv', '_EDA.xlsx')
writer = pd.ExcelWriter(output_filename_excel, engine='xlsxwriter')
workbook = writer.book

# Save EDA summary to Excel
stats_df = pd.DataFrame(stats).reset_index()
stats_df.columns = ['Statistic', 'Value']
stats_df.loc[len(stats_df)] = ['First Date', first_date] # Add first date to stats_df
stats_df.loc[len(stats_df)] = ['Last Date', last_date] # Add last date to stats_df
stats_df.loc[len(stats_df)] = ['Missing', miss]
stats_df.loc[len(stats_df)] = ['Normality p-value', norm_p]
stats_df.loc[len(stats_df)] = ['STL period used', period]
stats_df.loc[len(stats_df)] = ['Trend slope per step', slope]
stats_df.loc[len(stats_df)] = ['ADF Statistic', adf_test[0]]
stats_df.loc[len(stats_df)] = ['ADF p-value', adf_test[1]]
stats_df.loc[len(stats_df)] = ['Is stationary', 'Yes' if adf_test[1] <= 0.05 else 'No']


stats_df.to_excel(writer, sheet_name='EDA Summary', index=False)
worksheet_summary = writer.sheets['EDA Summary']

# List to store temporary filenames
temp_files = []

# Function to save plot to a temporary file and insert into Excel
def insert_plot_from_file(fig, sheet_name, cell, plot_name):
    """Saves a Matplotlib figure to a temporary file and inserts it into an Excel worksheet."""
    temp_filename = f"{plot_name}.png"
    fig.savefig(temp_filename)
    temp_files.append(temp_filename) # Add filename to list
    worksheet = writer.sheets[sheet_name]
    worksheet.insert_image(cell, temp_filename)
    plt.close(fig) # Close the figure to free up memory


# Plots and saving to Excel
# Histogram and Boxplot
fig, axes = plt.subplots(1, 3, figsize=(16,4))
sns.histplot(series.dropna(), kde=True, ax=axes[0]); axes[0].set_title('Histogram')
sns.boxplot(y=series.dropna(), ax=axes[1]); axes[1].set_title('Boxplot')
series.plot(ax=axes[2], label='Series'); axes[2].axhline(series.mean(), ls='--', label='Mean'); axes[2].set_title('Time Series with Mean')
axes[2].legend();
plt.tight_layout()
insert_plot_from_file(fig, 'EDA Summary', 'D2', 'hist_boxplot')


# Time series + Trend plot
if trend is not None:
    fig, ax = plt.subplots(figsize=(10,4))
    ax.plot(series.dropna(), alpha=.5, label='Original')
    ax.plot(trend, label=f'Trend (STL, s={period})', lw=2)
    pd.Series(series).rolling(30).mean().plot(ax=ax, label='Rolling mean (30D)')
    ax.set_title('Time series + Trend'); ax.legend();
    plt.tight_layout()
    insert_plot_from_file(fig, 'EDA Summary', 'D30', 'timeseries_trend')

# Rolling mean and std plot
fig, ax = plt.subplots(figsize=(10, 5))
rolling_mean = series.rolling(window=30).mean()
rolling_std = series.rolling(window=30).std()
ax.plot(series.index, series, label='Original')
ax.plot(rolling_mean.index, rolling_mean, label='Rolling Mean (30D)')
ax.plot(rolling_std.index, rolling_std, label='Rolling Std (30D)')
ax.set_title('Rolling Mean and Standard Deviation')
ax.set_xlabel('Date'); ax.set_ylabel('Value'); ax.legend();
plt.xticks(rotation=45)
plt.tight_layout()
insert_plot_from_file(fig, 'EDA Summary', 'D60', 'rolling_mean_std')


# ACF and PACF plots
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
plot_acf(series.dropna(), lags=50, ax=axes[0]); axes[0].set_title('Autocorrelation Function (ACF)')
plot_pacf(series.dropna(), lags=50, ax=axes[1], method='ywm'); axes[1].set_title('Partial Autocorrelation Function (PACF)')
plt.tight_layout()
insert_plot_from_file(fig, 'EDA Summary', 'D90', 'acf_pacf')

# STL Decomposition plots
if trend is not None:
    fig, axes = plt.subplots(4, 1, figsize=(10, 8))
    axes[0].plot(stl.observed); axes[0].set_title('Observed')
    axes[1].plot(stl.trend); axes[1].set_title('Trend')
    axes[2].plot(stl.seasonal); axes[2].set_title('Seasonal')
    axes[3].plot(stl.resid); axes[3].set_title('Residual')
    plt.tight_layout()
    insert_plot_from_file(fig, 'EDA Summary', 'D120', 'stl_decomposition')


# Close the Pandas Excel writer and save the Excel file
writer.close()

# Clean up temporary image files
for temp_file in temp_files:
    os.remove(temp_file)

print(f"\n📁 Saved EDA summary and plots to: {output_filename_excel}")


In [ ]:
# Exogenous entire set                  -> 2015-07-16 to 2025-08-06
# Biodiesel 100% price b100_gulf set    -> 2023-06-13 to 2025-08-05
# full_comsumption set                  -> 2016-01-01 to 2024-12-31 

In [ ]:
# === Cell 2: Updated for Rolling Cross-Validation and Residuals (Holt-Winters) ===

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.holtwinters import ExponentialSmoothing 
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
import io
from matplotlib.backends.backend_agg import FigureCanvasAgg as FigureCanvas
import xlsxwriter

# === Configuration ===
use_sqrt_transform = False  # <<< Set this to True to use sqrt transformation

# === Fold Control Parameters ===
n_splits = 8     # <<< Set number of rolling folds
test_size_days = 365   # <<< Set how many days each test fold should cover

# === Setup for TimeSeriesSplit ===
series = df[proxy_col]
series = series.sort_index()
series_dates = series.index

numeric_series = series.reset_index(drop=True)
tss = TimeSeriesSplit(n_splits=n_splits, test_size=test_size_days)
residuals_df = pd.DataFrame(index=series.index)
metrics_list = []

# === Step 2: Apply Transformation if Needed ===
if use_sqrt_transform:
    print("Applying Square-Root Transformation...")
    numeric_series = np.sqrt(numeric_series.clip(lower=0))
else:
    print("Using Original Data (No Transformation)...")
print("Using Raw Forecast (No Inverse Transformation)...")

# === Step 3: Train Holt-Winters Model In Rolling Folds ===
plt.figure(figsize=(12, 3 * n_splits))
last_forecast = None
last_test = None
last_params = None

for fold, (train_idx, test_idx) in enumerate(tss.split(numeric_series)):
    train = numeric_series.iloc[train_idx]
    test = numeric_series.iloc[test_idx]
    train_dates = series_dates[train_idx]
    test_dates = series_dates[test_idx]

    if len(train) < 100 or len(test) == 0:
        print(f"Skipping fold {fold} due to insufficient data")
        continue

    # === Holt-Winters Model Setup ===
    train = train.clip(lower=1e-3)  # Ensure strictly positive values for 'mul' seasonal
    start_time = time.time()
    trend_opt = "add"
    seasonal_opt = "add"
    seasonal_periods_opt = 180  # <-- user should manually change this as needed

    model = ExponentialSmoothing(
        train,
        trend=trend_opt,
        seasonal=seasonal_opt,
        seasonal_periods=seasonal_periods_opt
    ).fit()
    train_time = time.time() - start_time
    last_params = f"trend={trend_opt}, seasonal={seasonal_opt}, seasonal_periods={seasonal_periods_opt}"

    # === Step 4: Forecast on Test Period ===
    forecast = model.forecast(len(test))

    # === Step 5: Invert Transformation if Applied ===
    if use_sqrt_transform:
        forecast = forecast ** 2
        print("Applying Inverse Square-Root Transformation to Forecast...")

    # === Step 6: Calculate Metrics ===
    aligned_test = test
    aligned_forecast = forecast

    non_zero_test = aligned_test[aligned_test != 0]
    non_zero_forecast = aligned_forecast[aligned_test != 0]

    mae = mean_absolute_error(aligned_test, aligned_forecast)
    rmse = np.sqrt(mean_squared_error(aligned_test, aligned_forecast))
    mape = np.mean(np.abs((non_zero_test - non_zero_forecast) / non_zero_test)) * 100 if not non_zero_test.empty else np.nan
    r2 = r2_score(aligned_test, aligned_forecast)
    print(f"\n=== Metrics for Fold {fold} ({test_dates[0].date()} to {test_dates[-1].date()}) ===")
    print(f"MAE={mae:.4f}  RMSE={rmse:.4f}  MAPE={mape:.2f}%  R²={r2:.4f}")

    metrics_list.append({
        "Fold": fold,
        "Test Start": test_dates[0].date(),
        "Test End": test_dates[-1].date(),
        "MAE": mae,
        "RMSE": rmse,
        "MAPE": mape,
        "R2": r2,
        "SARIMA Params": last_params  # keeping column name for compatibility
    })

    # === Step 7: Residual Plot ===
    residual = aligned_test.values - aligned_forecast.values
    residual_series = pd.Series(residual, index=test_dates)
    residuals_df[f"fold_{fold}"] = residual_series

    plt.subplot(n_splits, 1, fold + 1)
    plt.plot(series.index, series.values, color='lightgray', label='_nolegend_')
    plt.plot(train_dates, train.values, label='Training Set', color='blue')
    plt.plot(test_dates, test.values, label='Test Set', color='red')
    plt.axvline(train_dates[-1], color='black', linestyle='--')
    plt.title(f"Data Train/Test Split Fold {fold}")
    plt.ylabel("Value")
    plt.xlabel("Date")
    plt.legend()

    last_forecast = aligned_forecast
    last_test = aligned_test

# === Step 8: Train/Test Split Visualization ===
plt.tight_layout()
plt.show()

# === Step 8.5: Print average metrics across all folds ===
if metrics_list:
    df_metrics = pd.DataFrame(metrics_list)
    mean_mae = df_metrics['MAE'].mean()
    mean_rmse = df_metrics['RMSE'].mean()
    mean_mape = df_metrics['MAPE'].mean()
    mean_r2 = df_metrics['R2'].mean()

    print("\n=== Mean Metrics Across All Folds ===")
    print(f"Mean MAE:  {mean_mae:.4f}")
    print(f"Mean RMSE: {mean_rmse:.4f}")
    print(f"Mean MAPE: {mean_mape:.2f}%")
    print(f"Mean R²:   {mean_r2:.4f}")
    print(f"trend_opt: {trend_opt}")
    print(f"seasonal_opt: {seasonal_opt}")
    print(f"seasonal_periods_opt: {seasonal_periods_opt}")
    print(f"n_splits: {n_splits}")

# === Residual Scatter Plot from Last Fold ===
residual_plot_buf = io.BytesIO()
if last_forecast is not None and last_test is not None:
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.residplot(x=last_forecast, y=(last_test - last_forecast), lowess=True,
                  scatter_kws={'alpha': 0.6, 'color': 'red'}, line_kws={'color': 'black'}, ax=ax)
    ax.set_title("Residual Plot (Holt-Winters) - Last Fold")
    ax.set_xlabel("Predicted values (Test Period)")
    ax.set_ylabel("Residuals (Test Period)")
    plt.tight_layout()
    fig.savefig(residual_plot_buf, format='png')
    residual_plot_buf.seek(0)
    plt.show()

# === Step 9: Combine All Residuals And Export to Excel ===
residuals_df["combined_residual"] = residuals_df.dropna(how='all', axis=1).bfill(axis=1).iloc[:, 0]
residuals_output = residuals_df.reset_index()
residuals_output.columns = ['date'] + [f for f in residuals_output.columns[1:]]
residuals_output_file = f"{proxy_col}_HW_residuals_by_fold.xlsx"

with pd.ExcelWriter(residuals_output_file, engine='xlsxwriter') as writer:
    residuals_output.to_excel(writer, sheet_name='Residuals', index=False)
    metrics_df = pd.DataFrame(metrics_list)
    metrics_df.to_excel(writer, sheet_name='Metrics', index=False)

    if last_forecast is not None:
        worksheet = writer.sheets['Metrics']
        worksheet.insert_image('J2', 'residual_last_fold.png', {'image_data': residual_plot_buf})

print(f"\n📁 Residuals by fold and last fold residual plot saved to: {residuals_output_file}")

In [ ]:
=== Mean Metrics Across All Folds ===
Mean MAE:  0.8876
Mean RMSE: 1.0664
Mean MAPE: 26.22%
Mean R²:   -16.9558
trend_opt: mul
seasonal_opt: add
seasonal_periods_opt: 180
n_splits: 8

=== Mean Metrics Across All Folds ===
Mean MAE:  0.9195
Mean RMSE: 1.1064
Mean MAPE: 27.29%
Mean R²:   -27.1139
trend_opt: mul
seasonal_opt: mul
seasonal_periods_opt: 180
n_splits: 8

=== Mean Metrics Across All Folds ===
Mean MAE:  0.7752
Mean RMSE: 0.9222
Mean MAPE: 22.62%
Mean R²:   -21.4751
trend_opt: add
seasonal_opt: mul
seasonal_periods_opt: 180
n_splits: 8

=== Mean Metrics Across All Folds ===
Mean MAE:  0.7670
Mean RMSE: 0.9083
Mean MAPE: 22.33%
Mean R²:   -15.1658
trend_opt: add
seasonal_opt: add
seasonal_periods_opt: 180
n_splits: 8

In [ ]:
# === Cell 3: Holt-Winters Model Training on FULL Data and Future Forecast ===

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.holtwinters import ExponentialSmoothing

# === Configuration ===
use_sqrt_transform_full = False            # keep True to match tuning
forecast_end_date = '2030-06-30'          # forecast horizon end
df.index = pd.to_datetime(df.index)       # ensure DateTimeIndex
df = df.sort_index()
train_start_date = '2016-01-01' #for exogenous ->'2015-07-16' // for b100_gulf ->'2023-06-13' // for TR -> '2016-01-01' # training window start

# === Step 1: Prepare full data series ===
df_train = df.loc[train_start_date:].copy().asfreq('D')
df_train[proxy_col] = df_train[proxy_col].ffill()

if use_sqrt_transform_full:
    y_full = np.sqrt(df_train[proxy_col].clip(lower=0))
    print("Applying Square-Root Transformation on full data...")
else:
    y_full = df_train[proxy_col]
    print("Using Original Data (No Transformation)...")

# === Step 2: Train Holt-Winters on selected dataset ===
start_time = time.time()
trend_opt = "add"
seasonal_opt = "add"
seasonal_periods_opt = 365

y_full = y_full.clip(lower=1e-3)  # Ensure positive for 'mul' if needed
model_full = ExponentialSmoothing(
    y_full,
    trend=trend_opt,
    seasonal=seasonal_opt,
    seasonal_periods=seasonal_periods_opt
).fit()

print(f"\nHolt-Winters Model trained on dataset from {train_start_date} ")
print(f"Model training time: {int(time.time() - start_time)} seconds")

# === Step 3: In-sample predictions & residuals ===
pred_mean = model_full.fittedvalues

if use_sqrt_transform_full:
    pred_inverse = pred_mean ** 2
    actual_inverse = df_train[proxy_col]
    print("Inverting transform for in-sample predictions...")
else:
    pred_inverse = pred_mean
    actual_inverse = df_train[proxy_col]

df_train["HW_fitted_in_sample"] = pred_inverse

df_train["sarima_predicted"] = pred_inverse
    # keep column name for consistency
df_train["sarima_residual"] = actual_inverse - pred_inverse


# === Step 4: Forecast future period ===
last_date = df.index[-1]
forecast_end_date = pd.to_datetime(forecast_end_date)
steps = (forecast_end_date - last_date).days
if steps <= 0:
    raise ValueError("forecast_end_date must be after the last observed date.")

future_forecast = model_full.forecast(steps)
future_forecast.index.freq = 'D'

if use_sqrt_transform_full:
    full_forecast = future_forecast ** 2
    print("Inverting transform for full forecast...")
else:
    full_forecast = future_forecast

# === Step 5: Confidence Intervals (simple fixed-width band) ===
z = 1.96
resid_std_full = np.std(df_train["sarima_residual"].dropna(), ddof=1)
ci_margin_full = z * resid_std_full
ci_lower_full = full_forecast - ci_margin_full
ci_upper_full = full_forecast + ci_margin_full

avg_ci_width_full = np.mean(ci_upper_full - ci_lower_full)
mean_pred_full = np.mean(full_forecast)
ci_width_pct_full = (avg_ci_width_full / mean_pred_full) * 100 if mean_pred_full != 0 else np.nan
print(f"\nAverage 95% CI Width (Fixed): {avg_ci_width_full:.4f} ({ci_width_pct_full:.2f}% of mean prediction)")

# === Step 6: Plot ===
plt.figure(figsize=(12, 6))
plt.plot(df.index, df[proxy_col].values, label='Actual')
plt.plot(full_forecast.index, full_forecast.values, label='Predicted (Full Horizon)')
plt.fill_between(full_forecast.index, ci_lower_full, ci_upper_full, color='gray', alpha=0.3, label='95% CI (Fixed)')
plt.title(f"Holt-Winters Forecast for {proxy_col}")
plt.ylabel(proxy_col)
plt.xlabel("Date")
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# === Cell 4: Save Results to XLSX ===

import xlsxwriter
import io
from matplotlib.backends.backend_agg import FigureCanvasAgg as FigureCanvas
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Sanity: required objects from Cell 3
assert 'full_forecast' in globals(), "full_forecast missing"
assert 'ci_lower_full' in globals() and 'ci_upper_full' in globals(), "CI arrays missing"

# Get Holt-Winters parameter string from Cell 3
hw_params_full = globals().get('hw_params_full', 'Unavailable')

# Fallbacks for CI summary if not in globals
avg_ci_width_full = globals().get('avg_ci_width_full', np.nan)
ci_width_pct_full = globals().get('ci_width_pct_full', np.nan)

# === Step 9: Prepare output DataFrames for XLSX ===
full_date_range_output = pd.date_range(start=df.index[0], end=full_forecast.index[-1], freq='D')

df_time_series_output = pd.DataFrame(index=full_date_range_output)
df_time_series_output['date'] = full_date_range_output
df_time_series_output[proxy_col] = df[proxy_col].reindex(full_date_range_output).values
df_time_series_output['predicted_future'] = full_forecast.reindex(full_date_range_output).values
df_time_series_output['ci_lower_future'] = ci_lower_full.reindex(full_date_range_output).values
df_time_series_output['ci_upper_future'] = ci_upper_full.reindex(full_date_range_output).values

# === ADD: HW fitted in-sample (history only) ===
df_time_series_output['HW_fitted_in_sample'] = (
    df_train['HW_fitted_in_sample']
    .reindex(full_date_range_output)
    .values
)
# === ADD: TargetXGB = actual - HW fitted (history only) ===
df_time_series_output['TargetXGB'] = (
    df_time_series_output[proxy_col]
    - df_time_series_output['HW_fitted_in_sample']
)


# Optional test-period outputs (only if variables exist from Cell 2)
residuals_test_aligned = globals().get('residuals_test', pd.Series(index=[], dtype=float)).reindex(full_date_range_output)
overlap_forecast_aligned = globals().get('overlap_forecast', pd.Series(index=[], dtype=float)).reindex(full_date_range_output)

df_time_series_output['test_residual'] = residuals_test_aligned.values
df_time_series_output['test_predicted'] = overlap_forecast_aligned.values
df_time_series_output['test_actual'] = df[proxy_col].reindex(full_date_range_output).values



# === Step 10: Metrics and Parameters Sheet (Sheet 2) ===
mae_val  = globals().get('mae',  np.nan)
rmse_val = globals().get('rmse', np.nan)
mape_val = globals().get('mape', np.nan)
r2_val   = globals().get('r2',   np.nan)

df_metrics_output = pd.DataFrame({
    'Metric': [
        'MAE (Test Set)',
        'RMSE (Test Set)',
        'MAPE (Test Set)',
        'R² (Test Set)',
        'Holt-Winters Parameters (Full)',
        'Average 95% CI Width (Full Forecast)',
        'Average 95% CI % (Full Forecast)'
    ],
    'Value': [
        mae_val,
        rmse_val,
        mape_val,
        r2_val,
        hw_params_full,
        avg_ci_width_full,
        ci_width_pct_full
    ]
})

# === Step 11: Save forecast plot to a buffer ===
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(df.index, df[proxy_col], label='Actual')
ax.plot(full_forecast.index, full_forecast, label='Forecast')
ax.fill_between(full_forecast.index, ci_lower_full, ci_upper_full, color='gray', alpha=0.3, label='95% CI')
ax.set_title(f"Holt-Winters Forecast with 95% CI for {proxy_col}")
ax.set_xlabel("Date"); ax.set_ylabel(proxy_col); ax.legend()
plt.xticks(rotation=45); plt.tight_layout()

plot_buffer = io.BytesIO()
plt.savefig(plot_buffer, format='png'); plt.close(fig); plot_buffer.seek(0)

# === Step 12: Write to Excel with embedded plot ===
output_filename = f"{proxy_col}_HoltWinters_Forecast_Full.xlsx"
with pd.ExcelWriter(output_filename, engine='xlsxwriter') as writer:
    df_time_series_output.to_excel(writer, sheet_name='Forecast Data', index=False)
    df_metrics_output.to_excel(writer, sheet_name='Metrics', index=False)
    worksheet = writer.sheets['Forecast Data']
    worksheet.insert_image('J2', 'forecast_plot.png', {'image_data': plot_buffer})

print(f"Final results saved to: {output_filename}")


In [ ]:
#Auxiliary codes:


In [ ]:
# === 3.5 Rolling "cluster-to-next-cluster" forecasts (8 rolls, 365 days each) ===
# Train on chunk 0 -> predict chunk 1
# Train on chunks 0..1 -> predict chunk 2
# ...
# Train on chunks 0..7 -> predict chunk 8
#
# Output:
# 1) Long plot: one fold per row
# 2) Excel: date + fold_0..fold_7 + stacked_prediction (first non-null across folds)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.holtwinters import ExponentialSmoothing

# =========================
# REQUIRED INPUTS (must exist)
# =========================
# df: DataFrame with DateTimeIndex
# proxy_col: string column name of the target series in df

# =========================
# USER SETTINGS
# =========================
n_splits = 8
test_size_days = 365

# Holt-Winters settings (robust defaults)
seasonal_periods = 365
# daily data => yearly seasonality
trend_opt = "add"

# =========================
# PREPARE SERIES
# =========================
df = df.copy()
df.index = pd.to_datetime(df.index)
df = df.sort_index()

y = df[proxy_col].asfreq("D").ffill()

# Split into consecutive chunks of equal length
chunk_len = test_size_days
needed_chunks = n_splits + 1
needed_len = needed_chunks * chunk_len

if len(y) < needed_len:
    raise ValueError(
        f"Not enough data. Need at least {needed_len} daily points "
        f"({needed_chunks} chunks of {chunk_len}). You have {len(y)}."
    )

y = y.iloc[:needed_len]  # exact length to fit chunks cleanly
chunks = [y.iloc[i*chunk_len:(i+1)*chunk_len] for i in range(needed_chunks)]

# Output table: one column per fold
preds_df = pd.DataFrame(index=y.index)
stacked = pd.Series(index=y.index, dtype=float)

# =========================
# PLOT SETUP (one fold per row)
# =========================
fig, axes = plt.subplots(n_splits, 1, figsize=(14, 3 * n_splits), sharex=True)
if n_splits == 1:
    axes = [axes]

# =========================
# ROLLING FORECASTS
# =========================
for fold in range(n_splits):
    train_series = pd.concat(chunks[:fold+1])     # chunk 0..fold
    test_series = chunks[fold+1]                 # chunk fold+1
    test_dates = test_series.index

    # Guard: HW seasonal needs enough cycles; else drop seasonality for early folds
    use_seasonal = ("add" if len(train_series) >= 2 * seasonal_periods else None)

    # Fit model
    train_fit = train_series.clip(lower=1e-3)  # keep numerically safe
    model = ExponentialSmoothing(
        train_fit,
        trend=trend_opt,
        seasonal=use_seasonal,
        seasonal_periods=(seasonal_periods if use_seasonal else None)
    ).fit(optimized=True)

    # Predict exactly the next chunk
    forecast = pd.Series(model.forecast(len(test_series)).values, index=test_dates)

    # Save fold column
    col = f"fold_{fold}"
    preds_df[col] = np.nan
    preds_df.loc[test_dates, col] = forecast.values

    # Plot fold (train blue, test red, forecast black)
    ax = axes[fold]
    ax.plot(train_series.index, train_series.values, label="Train", linewidth=1)
    ax.plot(test_series.index, test_series.values, label="Test", linewidth=1)
    ax.plot(forecast.index, forecast.values, label="Forecast", linewidth=1.5)
    ax.axvline(train_series.index[-1], color="black", linestyle="--")
    ax.set_title(f"Fold {fold}: train chunks 0..{fold} -> predict chunk {fold+1} (seasonal={use_seasonal})")
    ax.legend(loc="upper right")

plt.tight_layout()
plt.show()

# =========================
# STACKED COLUMN
# =========================
preds_df["stacked_prediction"] = preds_df[[f"fold_{i}" for i in range(n_splits)]].bfill(axis=1).iloc[:, 0]

# =========================
# EXPORT TO EXCEL
# =========================
out = preds_df.reset_index().rename(columns={"index": "date"})
output_file = f"{proxy_col}_rolling_prediction.xlsx"
out.to_excel(output_file, index=False)

print(f"Saved: {output_file}")


In [ ]:
# transforming full_comsumption in full_comsumption_weekly 
import pandas as pd

# === User Inputs ===
filename = "full_comsumption.csv"
target_col = "full_comsumption"

# === Load and Prepare Data ===
df = pd.read_csv(filename)

# Ensure datetime index
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df = df.dropna(subset=['date'])
df.set_index('date', inplace=True)

# Drop rows with NaN in the target column
df = df.dropna(subset=[target_col])

# === Weekly Aggregation ===
weekly_df = df[[target_col]].resample('W').sum()  # Use .mean() instead of .sum() if appropriate

# === Save Output ===
output_filename = filename.replace('.csv', '_weekly.csv')
weekly_df.to_csv(output_filename)

print(f"✅ Weekly data saved to: {output_filename}")